# Linux Inodes, Hard Links, and Symbolic Links

## A complete educational notebook

This notebook explains how Linux connects:

- filenames,
- pathnames,
- directories,
- inodes,
- data blocks,
- hard links,
- symbolic links,
- open files,
- filesystem boundaries,
- software version links,
- and practical commands.

The goal is to make the topic understandable even for a beginner.

---

## Learning objectives

After completing this notebook, you should be able to:

1. explain what an inode is,
2. distinguish a filename from a pathname,
3. explain where filenames are stored,
4. display inode numbers,
5. create and inspect hard links,
6. explain link counts,
7. describe what happens when a hard link is deleted,
8. explain why hard links cannot cross filesystems,
9. explain what a symbolic link stores,
10. create relative and absolute symbolic links,
11. identify broken symbolic links,
12. compare hard links, symbolic links, copies, and Windows shortcuts,
13. explain why Linux software often uses symbolic links,
14. use `basename`, `dirname`, `readlink`, `realpath`, `stat`, `find`, and `ls -li`.

# Part I — Files, Names, Paths, and Inodes

## 1. What is a file in Linux?

Linux uses the idea that many system resources can be accessed through file-like interfaces.

Examples include:

- regular files,
- directories,
- symbolic links,
- devices,
- pipes,
- sockets.

People often summarize this as:

> “Everything is a file.”

This is a useful idea, but not every object is literally an ordinary disk file.  
The important point is that Linux gives many resources a file-like interface.

This notebook focuses mainly on:

- regular files,
- directories,
- hard links,
- symbolic links.

## 2. Does every file have a name?

A beginner may say:

> “Every file has one name.”

That is not fully correct.

A Linux file may have:

- one filename,
- several filenames,
- or temporarily no filename at all.

A file can have several names because several directory entries may refer to the same inode. These are **hard links**.

A file can temporarily have no pathname if all directory entries are removed while a process still has it open.

So the more accurate statement is:

> A pathname identifies a directory entry, and that directory entry refers to an inode.

## 3. Filename, pathname, and path component

Consider:

```text
/home/ali/Documents/report.txt
```

### Filename

The final component is:

```text
report.txt
```

### Pathname

The complete path is:

```text
/home/ali/Documents/report.txt
```

### Path components

The components are:

```text
home
ali
Documents
report.txt
```

The slash `/` separates path components.

Linux does not store the complete pathname inside the inode.  
The pathname is resolved step by step through directories.

## 4. `basename` and `dirname`

The `basename` command extracts the final component of a pathname.

```bash
basename /home/ali/Documents/report.txt
```

Output:

```text
report.txt
```

The `dirname` command extracts the directory portion.

```bash
dirname /home/ali/Documents/report.txt
```

Output:

```text
/home/ali/Documents
```

Important:

- `basename` and `dirname` manipulate pathname text.
- They do not inspect the inode.
- The target does not even need to exist.

In [ ]:
basename /home/ali/Documents/report.txt
dirname /home/ali/Documents/report.txt
basename /a/b/c/
dirname /a/b/c/

## 5. What is an inode?

An **inode** is a filesystem data structure that stores metadata about a filesystem object.

Typical inode information includes:

- inode number,
- file type,
- owner UID,
- group GID,
- permission bits,
- file size,
- timestamps,
- hard-link count,
- pointers or references to data blocks,
- extended metadata depending on the filesystem.

### Important correction

It is not correct to say:

> “All information about a file is stored in the inode.”

The inode stores metadata and references to file data.  
The actual file contents are usually stored in data blocks.  
The filename is stored in a directory entry, not in the inode.

A better model is:

```text
Directory entry:
    filename  ---> inode number

Inode:
    metadata + references to data blocks

Data blocks:
    actual file contents
```

## 6. Simplified filesystem picture

```text
Directory: /home/ali

report.txt  ------------+
photo.jpg   --------+   |
                    |   |
                    v   v

inode 812        inode 905
metadata         metadata
size             size
permissions      permissions
data pointers    data pointers
    |                |
    v                v
data blocks      data blocks
```

The directory stores the names and inode references.

The inode stores metadata.

The data blocks store the contents.

## 7. Where is the filename stored?

A directory is a special filesystem object containing entries similar to:

```text
filename  -> inode number
```

Conceptually:

```text
report.txt  -> inode 812
notes.txt   -> inode 930
photo.jpg   -> inode 905
```

The filename belongs to the directory entry.

This explains why:

- renaming a file usually changes a directory entry,
- deleting a filename removes a directory entry,
- one inode can have several names,
- an inode does not know one unique “official name.”

## 8. One name and one inode

Within one directory, one filename can identify only one directory entry at a time.

Therefore, a specific name such as:

```text
report.txt
```

cannot simultaneously point to two different inodes in the same directory.

However, the same text name can appear in different directories:

```text
/home/ali/report.txt
/tmp/report.txt
```

These are different directory entries and may refer to different inodes.

## 9. One inode and several names

One inode may have several directory entries pointing to it.

Example:

```text
/home/ali/original.txt  ----+
                            |
/home/ali/hardlink.txt  ----+----> inode 2001
```

These two names are equal references to the same underlying file.

There is no special “original” name at the filesystem level.

# Part II — Inspecting Inodes

## 10. How can we see an inode number?

Use:

```bash
ls -i filename
```

Example:

```text
2001 report.txt
```

For detailed output:

```bash
ls -li filename
```

The `-i` option displays the inode number.

You can also use:

```bash
stat filename
```

In [ ]:
rm -rf inode_link_lab
mkdir inode_link_lab
cd inode_link_lab

printf "Hello Linux\n" > report.txt

ls -i report.txt
ls -li report.txt
stat report.txt

## 11. Understanding relevant `stat` fields

Example:

```text
File: report.txt
Size: 12
Blocks: 8
IO Block: 4096 regular file
Device: ...
Inode: 2001
Links: 1
Access: (0644/-rw-r--r--)
Uid: (1000/ali)
Gid: (1000/ali)
Access: ...
Modify: ...
Change: ...
Birth: ...
```

Important fields:

| Field | Meaning |
|---|---|
| File | pathname given to `stat` |
| Size | logical file size |
| Blocks | allocated blocks |
| Inode | inode number |
| Links | hard-link count |
| Access | permissions |
| Uid | owner |
| Gid | group |
| Access time | last access time |
| Modify time | content modification time |
| Change time | inode metadata change time |
| Birth time | creation time, if supported |

`stat` displays information about the target by default.

To inspect the symbolic link itself later, use:

```bash
stat -c ...
```

or:

```bash
lstat
```

through programming interfaces. GNU `stat` normally reports the symlink itself when passed a symlink unless dereferencing is requested with `-L`.

## 12. Link count in `ls -l`

Consider:

```text
-rw-r--r-- 1 ali ali 12 Jul 10 10:00 report.txt
```

The number after the permissions is:

```text
1
```

For a regular file, this is normally the number of hard links to the inode.

After creating another hard link, it may become:

```text
-rw-r--r-- 2 ali ali 12 Jul 10 10:00 report.txt
-rw-r--r-- 2 ali ali 12 Jul 10 10:00 second_name.txt
```

Both names show link count `2`.

# Part III — Hard Links

## 13. What is a hard link?

A hard link is another directory entry that points to the same inode.

Create one with:

```bash
ln existing_file new_name
```

Example:

```bash
ln report.txt report-hard.txt
```

Now both names refer to the same inode.

In [ ]:
ln report.txt report-hard.txt

ls -li report.txt report-hard.txt
stat -c 'name=%n inode=%i links=%h size=%s' report.txt report-hard.txt

## 14. There is no original and copy

After:

```bash
ln report.txt report-hard.txt
```

it is tempting to say:

- `report.txt` is the original,
- `report-hard.txt` is the link.

At the filesystem level, this distinction does not exist.

Both are ordinary directory entries pointing to the same inode:

```text
report.txt       ---+
                   +---> inode 2001
report-hard.txt  ---+
```

They are equal names for the same file.

## 15. What happens when one hard-link name is changed?

There are two different meanings of “change.”

### Changing file contents

If you append through one name:

```bash
echo "new line" >> report-hard.txt
```

then reading through the other name shows the same content:

```bash
cat report.txt
```

Why?

Because both names refer to the same inode and the same data blocks.

### Renaming one directory entry

If you run:

```bash
mv report-hard.txt second-name.txt
```

only that pathname changes.

The inode and contents remain the same.

In [ ]:
printf "Added using hard link\n" >> report-hard.txt

echo "Content through report.txt:"
cat report.txt

echo
echo "Content through report-hard.txt:"
cat report-hard.txt

mv report-hard.txt second-name.txt
ls -li report.txt second-name.txt

## 16. Do hard links share metadata?

Because hard links refer to the same inode, they share inode metadata such as:

- owner,
- group,
- permissions,
- size,
- modification time,
- link count.

For example:

```bash
chmod 600 second-name.txt
```

will also appear when inspecting:

```bash
ls -l report.txt
```

because both names lead to the same inode.

In [ ]:
chmod 600 second-name.txt
ls -li report.txt second-name.txt

## 17. What happens if one hard link is deleted?

Suppose the inode has two names:

```text
report.txt
second-name.txt
```

Running:

```bash
rm second-name.txt
```

removes only that directory entry.

The inode still has one remaining hard link:

```text
report.txt
```

The file contents are still available.

In [ ]:
rm second-name.txt
ls -li report.txt
stat -c 'inode=%i links=%h name=%n' report.txt
cat report.txt

## 18. When is the inode actually removed?

A file's storage is normally reclaimed when:

1. the hard-link count becomes zero, and
2. no running process still has the file open.

This is more precise than saying:

> “The file disappears immediately after `rm`.”

The `rm` command removes a directory entry.  
It does not directly erase the data blocks.

## 19. Open but deleted files

Linux allows a process to keep using a file after its pathname has been removed.

Conceptual example:

1. a process opens `log.txt`,
2. another process runs `rm log.txt`,
3. the directory entry disappears,
4. the first process still has an open file descriptor,
5. the inode and data remain until the process closes the file.

This behavior explains situations where:

- disk space remains occupied after deleting a large log,
- restarting a service releases disk space,
- upgraded executables can coexist temporarily with running old processes.

Useful command:

```bash
lsof +L1
```

This may show open files whose link count is zero.

## 20. Practical applications of hard links

Hard links are useful for:

### Multiple names

The same data may be accessible under different names.

### Space-efficient backup systems

A backup can hard-link unchanged files from a previous backup while storing changed files separately.

### Package management and deduplication

Some tools use hard links to avoid storing identical content twice.

### Atomic replacement patterns

Programs may create a new file and replace a directory entry while existing processes continue using the old inode.

### Preserving data while reorganizing names

A new name can be created before an old name is removed.

Important:

Hard links are not copies.  
They refer to the same underlying data.

## 21. Hard link versus copy

Suppose:

```bash
cp report.txt copied.txt
ln report.txt linked.txt
```

### Copy

```text
report.txt  -> inode 100
copied.txt  -> inode 200
```

Different files and different data blocks.

### Hard link

```text
report.txt  ---+
               +--> inode 100
linked.txt  ---+
```

Same inode and same data.

Changing `copied.txt` does not change `report.txt`.

Changing `linked.txt` changes the same file seen through `report.txt`.

In [ ]:
cp report.txt copied.txt
ln report.txt linked.txt

ls -li report.txt copied.txt linked.txt

printf "Only in copy\n" >> copied.txt
printf "Shared through hard link\n" >> linked.txt

echo "--- report.txt ---"
cat report.txt
echo "--- copied.txt ---"
cat copied.txt

# Part IV — Restrictions of Hard Links

## 22. Why can hard links not cross filesystems?

A hard link stores a reference to an inode in the same filesystem.

Each filesystem manages its own inode namespace.

For example:

```text
Filesystem A:
inode 100
inode 101

Filesystem B:
inode 100
inode 101
```

The number `100` in filesystem A is unrelated to inode `100` in filesystem B.

A directory entry in filesystem A cannot directly point to an inode owned by filesystem B.

Therefore:

```bash
ln /filesystem-A/file /filesystem-B/link
```

normally fails with:

```text
Invalid cross-device link
```

This is why people often say hard links cannot cross partitions.  
More precisely, they cannot cross filesystem boundaries.

## 23. Partition versus filesystem

A partition is a region of a storage device.

A filesystem is the structure placed on storage to organize files.

Often one partition contains one filesystem, but not always.

Hard-link restrictions are about **filesystems**, not merely physical disks.

For example:

- two mount points may be on the same disk but different filesystems,
- two devices may participate in one logical filesystem,
- a network mount is a different filesystem boundary.

## 24. Why ordinary users cannot hard-link directories

Hard links to directories could create cycles.

Example:

```text
A/
└── B/
    └── link-to-A/
```

If `link-to-A` were a hard link to directory `A`, the directory tree would contain a loop.

Problems would include:

- infinite recursive traversal,
- difficult link-count management,
- broken `..` relationships,
- filesystem-checking complexity,
- backup tools looping forever.

Therefore, Linux normally forbids ordinary users from creating hard links to directories.

The filesystem itself uses directory links internally, including:

- `.`
- `..`

but users should not create arbitrary directory hard links.

## 25. Hard-link security restrictions

Modern Linux systems may also restrict creating a hard link to a file you do not own.

A relevant kernel setting is:

```bash
sysctl fs.protected_hardlinks
```

When enabled, it helps prevent certain attacks where one user hard-links another user's file and tricks a privileged program into modifying it.

Therefore, even within the same filesystem, permission and security policies may prevent a hard link.

In [ ]:
sysctl fs.protected_hardlinks 2>/dev/null || true

# Part V — Symbolic Links

## 26. What is a symbolic link?

A symbolic link, also called a **symlink** or **soft link**, is a separate filesystem object that stores a pathname.

Create one with:

```bash
ln -s target link_name
```

Example:

```bash
ln -s report.txt report-shortcut
```

Conceptually:

```text
report-shortcut
    inode 300
    contents: "report.txt"

report.txt
    inode 100
    contents: actual text
```

The symlink and target have different inodes.

In [ ]:
ln -s report.txt report-shortcut

ls -li report.txt report-shortcut
stat -c 'name=%n inode=%i type=%F links=%h' report.txt report-shortcut
readlink report-shortcut

## 27. Link and target

A symbolic-link relationship has:

- the **link**: the symlink object,
- the **target**: the pathname stored inside the link.

Example:

```text
report-shortcut -> report.txt
```

Here:

- link = `report-shortcut`
- stored target pathname = `report.txt`
- final target object = the file reached by resolving `report.txt`

## 28. Does a symbolic link point directly to an inode?

No.

A symbolic link stores a pathname, not a permanent inode reference.

Suppose:

```text
shortcut -> report.txt
```

When a program opens `shortcut`, the kernel reads the stored pathname `report.txt` and resolves it.

This is why symlinks:

- can cross filesystems,
- can point to directories,
- may become broken,
- may start referring to a different inode if the target path is replaced.

## 29. What happens when the target is deleted?

Suppose:

```text
shortcut -> target.txt
```

If `target.txt` is removed, the symlink still exists.

The symlink becomes a **broken** or **dangling** symbolic link.

Example output:

```text
lrwxrwxrwx shortcut -> target.txt
```

but opening it fails:

```text
No such file or directory
```

The symlink itself has not disappeared.  
Its stored pathname no longer resolves to a valid object.

In [ ]:
printf "temporary target\n" > temp-target.txt
ln -s temp-target.txt temp-link

ls -l temp-link
cat temp-link

rm temp-target.txt

ls -l temp-link
cat temp-link 2>&1 || true

## 30. Recreating the target repairs the symlink

A broken symlink stores a pathname.

If a new file is later created at that pathname:

```bash
echo "new target" > temp-target.txt
```

then the same symlink works again.

The new target may have a completely different inode.

This proves the symlink follows the pathname rather than permanently remembering the old inode.

In [ ]:
printf "new target object\n" > temp-target.txt
ls -li temp-target.txt temp-link
cat temp-link

## 31. Absolute symbolic links

An absolute symlink stores a pathname beginning with `/`.

Example:

```bash
ln -s /home/ali/data/report.txt current-report
```

Advantages:

- clear destination,
- works regardless of the symlink's containing directory.

Disadvantages:

- may break if the directory tree is moved,
- less portable between systems,
- may not work inside containers or chroot environments.

## 32. Relative symbolic links

A relative symlink stores a relative pathname.

Example:

```bash
ln -s ../data/report.txt current-report
```

The path is interpreted relative to the directory containing the symlink.

Advantages:

- can remain valid when an entire directory tree is moved,
- useful in software packages and project directories.

Common mistake:

A relative target is not interpreted relative to the current shell directory after creation.  
It is interpreted relative to the symlink's location when the link is followed.

## 33. Relative-link example

Directory structure:

```text
project/
├── data/
│   └── report.txt
└── links/
    └── current -> ../data/report.txt
```

The link command can be:

```bash
cd project/links
ln -s ../data/report.txt current
```

The stored path is:

```text
../data/report.txt
```

From the `links` directory, this correctly reaches `data/report.txt`.

In [ ]:
mkdir -p relative_demo/project/data
mkdir -p relative_demo/project/links

printf "relative symlink works\n" > relative_demo/project/data/report.txt
ln -s ../data/report.txt relative_demo/project/links/current

ls -l relative_demo/project/links/current
readlink relative_demo/project/links/current
realpath relative_demo/project/links/current
cat relative_demo/project/links/current

## 34. Symlinks to directories

A symbolic link may point to a directory:

```bash
ln -s /var/log logs
```

Then:

```bash
cd logs
```

takes you to the target directory.

This is a common advantage over hard links.

Applications include:

- exposing data in a convenient location,
- redirecting application directories,
- versioned deployment directories,
- configuration management.

## 35. Symlink permissions

A symbolic link often appears as:

```text
lrwxrwxrwx
```

These displayed permissions are usually not used in the ordinary way.

Access checks normally apply to:

- directories traversed during path resolution,
- the target object.

On Linux, changing symlink permissions is generally not meaningful in the usual way.

Ownership of a symlink can matter in some security-sensitive directory operations, especially with sticky directories and protected symlink settings.

# Part VI — Useful Commands

## 36. `readlink`

`readlink` displays the pathname stored inside a symbolic link.

```bash
readlink shortcut
```

Example:

```text
../data/report.txt
```

It does not necessarily show the final canonical target.

## 37. `realpath`

`realpath` resolves:

- relative path components,
- `.` and `..`,
- symbolic links,
- and prints a canonical absolute path.

Example:

```bash
realpath shortcut
```

Possible output:

```text
/home/ali/project/data/report.txt
```

If the target does not exist, behavior depends on options and implementation.

## 38. `find -inum`

To find names referring to a known inode:

```bash
find . -inum 12345
```

Example workflow:

```bash
inode=$(stat -c %i report.txt)
find . -inum "$inode"
```

This can locate hard links within the searched filesystem.

Remember:

- inode numbers are meaningful only together with the filesystem/device,
- the same inode number may exist on another filesystem.

In [ ]:
inode_number=$(stat -c %i report.txt)
echo "inode=$inode_number"
find . -xdev -inum "$inode_number" -print

## 39. Finding symbolic links

Find symbolic links:

```bash
find . -type l
```

Show stored targets:

```bash
find . -type l -exec ls -l {} \;
```

Find broken symbolic links:

```bash
find . -xtype l
```

or:

```bash
find -L . -type l
```

The exact behavior differs between `find` expressions and link-following options, so test carefully.

In [ ]:
echo "All symbolic links:"
find . -type l -print

echo
echo "Potential broken symbolic links:"
find . -xtype l -print 2>/dev/null || true

## 40. `ls -l`, `ls -L`, and `ls -H`

### `ls -l link`

Shows the symlink itself:

```text
link -> target
```

### `ls -lL link`

Dereferences the symlink and shows target information.

### `ls -lH link`

Dereferences command-line symlink arguments but may treat encountered symlinks differently during recursion.

These options matter when inspecting trees containing links.

## 41. Removing symbolic links safely

To remove a symbolic link:

```bash
rm link_name
```

This removes the link, not the target.

You may also use:

```bash
unlink link_name
```

Be careful with a trailing slash:

```bash
rm symlink-to-directory/
```

Behavior may differ or fail because the slash requests directory treatment.

Safer:

```bash
rm symlink-to-directory
```

# Part VII — Hard Links vs Symbolic Links vs Copies

## 42. Comprehensive comparison table

| Property | Hard link | Symbolic link | Copy |
|---|---|---|---|
| Same inode as source | Yes | No | No |
| Stores pathname | No | Yes | No |
| Shares file contents | Yes | Accesses target | No |
| Crosses filesystems | No | Yes | Yes |
| Can point to directory | Normally no | Yes | Copy creates a new directory tree |
| Survives removal of another name | Yes | No, if that was target path | Yes |
| Can become broken | No | Yes | No |
| Link count increases | Yes | No | No |
| Different permissions | No, same inode | Link and target are separate | Yes |
| Uses extra data storage | Minimal directory entry | Small symlink object | Full duplicated data |
| Editing affects source | Yes | Usually yes, through target | No |

## 43. Is a symbolic link the same as a Windows shortcut?

It is similar in purpose but not identical.

### Similarity

Both provide an indirect way to reach another object.

### Difference

A Windows `.lnk` shortcut is usually interpreted by the graphical shell or applications.

A Unix symbolic link is part of filesystem pathname resolution.  
Most programs transparently follow it without special shortcut logic.

Modern Windows also supports true symbolic links and hard links through NTFS, which are closer to Linux links than `.lnk` shortcut files.

# Part VIII — Real-World Applications

## 44. Python version management

Linux systems may contain several Python executable names:

```text
python
python3
python3.12
```

A possible chain is:

```text
python -> python3
python3 -> python3.12
```

This lets administrators change which version a general command points to without replacing every script or command.

Inspect with:

```bash
command -v python
command -v python3
ls -l "$(command -v python3)"
readlink -f "$(command -v python3)"
```

Important:

Not every distribution provides `python`, and not every Python command is a symlink.  
Some may be wrapper programs, alternatives-managed links, virtual-environment executables, or real binaries.

In [ ]:
for cmd in python python3; do
    if command -v "$cmd" >/dev/null 2>&1; then
        path=$(command -v "$cmd")
        echo "$cmd -> $path"
        ls -l "$path"
        realpath "$path" 2>/dev/null || true
        echo
    fi
done

## 45. Python virtual environments

A Python virtual environment often contains:

```text
venv/bin/python
venv/bin/python3
venv/bin/pip
```

Depending on the platform and tool, these may include symlinks pointing to a base Python interpreter.

The environment also changes import paths and installed package locations.

A symlink alone is not the entire virtual environment mechanism, but symlinks may be part of its layout.

## 46. Shared libraries

Shared libraries commonly use versioned symlink chains.

Example:

```text
libexample.so -> libexample.so.2
libexample.so.2 -> libexample.so.2.4.1
libexample.so.2.4.1
```

Roles:

- unversioned name: often used during compilation,
- major-version name: ABI compatibility name,
- full-version file: actual library implementation.

This lets software request a compatible major version while administrators update minor releases.

## 47. `/etc/alternatives`

Some Linux distributions use an alternatives system.

Example concept:

```text
/usr/bin/editor
    -> /etc/alternatives/editor
    -> /usr/bin/vim.basic
```

This allows a system-wide default program to be selected without manually replacing every command name.

Useful commands on Debian/Ubuntu systems:

```bash
update-alternatives --display editor
update-alternatives --config editor
```

Other distributions may use different tools.

## 48. Current-release deployment links

Applications may use:

```text
/opt/myapp/releases/2026-07-01/
/opt/myapp/releases/2026-07-10/
/opt/myapp/current -> releases/2026-07-10
```

To deploy a new version, the `current` symlink can be replaced.

Benefits:

- quick switching,
- easy rollback,
- stable path for services,
- versioned releases remain separate.

For reliable updates, administrators often create a new symlink and rename it atomically.

## 49. Redirecting large data directories

Suppose an application expects:

```text
/home/ali/data
```

but the data must live on another mounted filesystem:

```text
/mnt/storage/data
```

One approach:

```bash
mv /home/ali/data /mnt/storage/data
ln -s /mnt/storage/data /home/ali/data
```

The application can continue using the old pathname.

However, consider:

- permissions,
- mount availability,
- backup behavior,
- service startup order,
- container visibility,
- absolute-path portability.

## 50. Backup tools and hard links

Snapshot-style backup tools may create directories such as:

```text
backup-1/
backup-2/
backup-3/
```

Unchanged files in `backup-2` may be hard links to files in `backup-1`.

To users, every backup directory looks complete.

On disk, unchanged data may be stored only once.

When a changed file is backed up, a new inode is created for the new version.

# Part IX — Commands That Interact with Links

## 51. `cp` and links

Copy behavior depends on options and context.

Common GNU `cp` options:

| Option | Meaning |
|---|---|
| `-P` | do not follow source symlinks |
| `-L` | follow source symlinks |
| `-H` | follow command-line symlink arguments |
| `-a` | archive mode; preserve links and metadata as much as possible |
| `-d` | preserve symlinks and hard-link relationships where possible |

Always verify behavior when copying directory trees containing links.

## 52. `tar` and symbolic links

By default, `tar` usually archives a symbolic link as a symbolic link.

With dereference options, it may archive the target contents instead.

Why this matters:

- preserving deployment layouts,
- avoiding unexpectedly huge archives,
- preventing traversal into external directory trees,
- restoring links correctly.

## 53. `rsync` and links

Important `rsync` options include:

| Option | Meaning |
|---|---|
| `-l` | copy symlinks as symlinks |
| `-L` | transform symlinks into copied referents |
| `-H` | preserve hard links |
| `-a` | archive mode; includes symlink preservation but not always hard-link preservation |
| `--safe-links` | ignore unsafe links pointing outside copied tree |
| `--copy-unsafe-links` | copy referents of unsafe links |

Hard-link preservation can consume memory on very large trees.

## 54. `mv` and filesystems

Within one filesystem, renaming or moving a file often updates directory entries without copying file contents.

Across filesystems, `mv` generally behaves more like:

1. copy,
2. preserve metadata as possible,
3. remove source.

Therefore, an inode number normally changes when moving across filesystems.

# Part X — Directory Link Counts

## 55. Why do directories have link counts?

For a directory, link counts are more complex than for ordinary files.

Traditionally, a directory's link count is related to:

- its own `.` entry,
- the parent directory's entry naming it,
- each immediate subdirectory's `..` entry.

For a simple directory with no subdirectories, the link count is often `2`.

Adding a subdirectory may increase the parent directory's link count.

However, some modern filesystems optimize directory link counts or report special values, so this rule is not universal.

In [ ]:
mkdir -p dir_link_count_demo
echo "Before subdirectory:"
ls -ld dir_link_count_demo

mkdir dir_link_count_demo/child
echo "After subdirectory:"
ls -ld dir_link_count_demo

ls -lid dir_link_count_demo dir_link_count_demo/. dir_link_count_demo/child/..

# Part XI — Security Considerations

## 56. Symbolic-link attacks

A symbolic-link attack can occur when a privileged program:

1. uses a predictable pathname in a writable directory,
2. fails to safely create or open the file,
3. follows an attacker-controlled symlink,
4. writes to an unintended target.

Safer programming practices include:

- secure temporary-file APIs,
- exclusive file creation,
- avoiding predictable names,
- checking ownership and file type,
- using directory file descriptors,
- using flags such as `O_NOFOLLOW` where appropriate.

## 57. Protected symlinks

Linux may use:

```bash
sysctl fs.protected_symlinks
```

This hardening feature helps restrict unsafe symlink following in sticky world-writable directories such as `/tmp`.

It is not a substitute for secure application design.

In [ ]:
sysctl fs.protected_symlinks 2>/dev/null || true

## 58. Hard-link attacks

Hard-link restrictions help prevent an attacker from creating a link to a sensitive file and waiting for a privileged program to modify that pathname.

Relevant setting:

```bash
sysctl fs.protected_hardlinks
```

This is one reason modern Linux may refuse hard-link creation even when both paths are on the same filesystem.

# Part XII — Complete Hands-On Lab

## 59. Lab setup

The following lab operates only inside a new directory.

It does not require root.

In [ ]:
cd ..
rm -rf complete_links_lab
mkdir complete_links_lab
cd complete_links_lab

printf "Line 1\n" > alpha.txt
printf "Line 2\n" >> alpha.txt

echo "Initial file:"
ls -li alpha.txt
stat -c 'name=%n inode=%i links=%h size=%s mode=%A' alpha.txt

## 60. Lab: create a hard link

In [ ]:
ln alpha.txt beta.txt

ls -li alpha.txt beta.txt
stat -c 'name=%n inode=%i links=%h size=%s' alpha.txt beta.txt

### Observation

Both names should show:

- the same inode number,
- the same size,
- link count `2`.

## 61. Lab: modify through the second name

In [ ]:
printf "Added through beta\n" >> beta.txt

echo "--- alpha.txt ---"
cat alpha.txt

echo "--- beta.txt ---"
cat beta.txt

### Observation

Both commands display the same contents because the names refer to one inode.

## 62. Lab: compare with a copy

In [ ]:
cp alpha.txt gamma.txt

ls -li alpha.txt beta.txt gamma.txt

printf "Only gamma changes\n" >> gamma.txt

echo "--- alpha.txt ---"
cat alpha.txt

echo "--- gamma.txt ---"
cat gamma.txt

### Observation

`gamma.txt` has a different inode and independent contents.

## 63. Lab: remove one hard link

In [ ]:
rm beta.txt

ls -li alpha.txt
stat -c 'name=%n inode=%i links=%h' alpha.txt

### Observation

`alpha.txt` remains.  
The link count decreases from `2` to `1`.

## 64. Lab: create a symbolic link

In [ ]:
ln -s alpha.txt alpha-link

ls -li alpha.txt alpha-link
readlink alpha-link
realpath alpha-link
cat alpha-link

### Observation

The symlink has:

- a different inode,
- its own file type `l`,
- stored pathname `alpha.txt`.

## 65. Lab: break and repair a symbolic link

In [ ]:
mv alpha.txt alpha-renamed.txt

echo "After renaming the target:"
ls -l alpha-link
cat alpha-link 2>&1 || true

mv alpha-renamed.txt alpha.txt

echo
echo "After restoring the target pathname:"
cat alpha-link

### Observation

The link broke when the target pathname disappeared.

It worked again when a file returned at the stored pathname.

## 66. Lab: absolute and relative links

In [ ]:
mkdir -p tree/data tree/links
printf "project data\n" > tree/data/item.txt

ln -s ../data/item.txt tree/links/relative
ln -s "$(pwd)/tree/data/item.txt" tree/links/absolute

ls -l tree/links
readlink tree/links/relative
readlink tree/links/absolute

cat tree/links/relative
cat tree/links/absolute

## 67. Lab: move the entire tree

A relative link inside the tree often survives moving the tree.

An absolute link may continue pointing to the old location and become broken.

In [ ]:
mv tree moved-tree

echo "Relative link:"
ls -l moved-tree/links/relative
cat moved-tree/links/relative

echo
echo "Absolute link:"
ls -l moved-tree/links/absolute
cat moved-tree/links/absolute 2>&1 || true

# Part XIII — Common Mistakes

## 68. Common mistake: saying a hard link points to another filename

Incorrect mental model:

```text
hardlink -> original filename
```

Better model:

```text
name A ---+
          +--> same inode
name B ---+
```

A hard link does not depend on another pathname.

## 69. Common mistake: saying the inode contains the filename

The filename is stored in a directory entry.

The inode stores metadata and file-data references.

An inode may have several names, so it cannot store one unique ordinary filename.

## 70. Common mistake: assuming a symlink stores the target inode

A symlink stores pathname text.

Replacing the target path with a new file causes the symlink to follow the new inode.

## 71. Common mistake: assuming deletion erases data immediately

`rm` removes a directory entry.

Data is reclaimed only when:

- no hard links remain,
- no process keeps the inode open.

## 72. Common mistake: using `ln -s` arguments backwards

Syntax:

```bash
ln -s TARGET LINK_NAME
```

Example:

```bash
ln -s /opt/app/releases/v2 /opt/app/current
```

Read it as:

> Create `current`, which points to `releases/v2`.

# Part XIV — Review Questions

## 73. Questions

1. What is an inode?
2. Is a filename stored inside the inode?
3. Where is a filename stored?
4. What is the difference between a filename and pathname?
5. What do `basename` and `dirname` do?
6. Which command displays inode numbers?
7. What does the `Links` field in `stat` mean?
8. Can one inode have several filenames?
9. Can one directory entry point to two inodes?
10. How do you create a hard link?
11. Does a hard link have an original and a copy?
12. Why does editing one hard link affect the other?
13. What happens when one hard-link name is deleted?
14. When are the inode and data blocks finally reclaimed?
15. Why can hard links not cross filesystems?
16. Why are directory hard links restricted?
17. What does a symbolic link store?
18. Do a symlink and target have the same inode?
19. What happens when a symbolic-link target is deleted?
20. Can a broken symlink work again?
21. Can a symlink cross filesystems?
22. Can a symlink point to a directory?
23. What is the difference between relative and absolute symlinks?
24. Why are relative symlinks useful in movable directory trees?
25. How is a Linux symlink different from a Windows `.lnk` shortcut?
26. Why are symlinks used for Python commands?
27. Why are symlinks used for shared libraries?
28. What does `readlink` show?
29. What does `realpath` show?
30. How can you find all names with a particular inode?
31. How can you find broken symlinks?
32. What is the difference between a hard link and a copy?
33. Why might disk space remain used after deleting a large file?
34. What can `lsof +L1` help find?
35. What security risk can unsafe symbolic-link handling create?

# Part XV — Answers

## 74. Answers

1. An inode is a filesystem structure storing metadata and references to file data.
2. No.
3. In a directory entry.
4. A filename is one path component; a pathname describes how to locate an object through directories.
5. They extract the last component and directory portion of pathname text.
6. `ls -i`, `ls -li`, or `stat`.
7. The number of hard-link directory entries referring to the inode.
8. Yes.
9. No, one entry resolves to one inode at a time.
10. `ln existing_file new_name`.
11. No. Both names are equal references to the inode.
12. They refer to the same inode and data blocks.
13. Only that directory entry is removed; other hard links continue working.
14. When link count is zero and no process has the file open.
15. Inodes belong to a particular filesystem's namespace.
16. They could create cycles and break directory-tree assumptions.
17. A pathname.
18. No.
19. The symlink becomes broken.
20. Yes, if a valid object later appears at the stored target pathname.
21. Yes.
22. Yes.
23. An absolute target begins at `/`; a relative target is interpreted from the symlink's containing directory.
24. They may continue working when the whole tree moves together.
25. A Linux symlink is resolved by filesystem pathname handling; `.lnk` files are usually shell-level shortcuts.
26. To select or expose a convenient interpreter name while preserving versioned executables.
27. To connect generic, ABI-major, and full-version library names.
28. The pathname stored inside the symlink.
29. The resolved canonical pathname.
30. `find ... -inum NUMBER`.
31. For example, `find . -xtype l`.
32. A hard link shares the inode; a copy has a new inode and independent data.
33. A process may still have the deleted inode open.
34. Open files whose link count has become zero.
35. A privileged program may be tricked into reading or writing an unintended target.

# Part XVI — Final Cheat Sheet

## 75. Cheat sheet

### Inode inspection

```bash
ls -i file
ls -li file
stat file
stat -c '%n %i %h %F' file
```

### Path text

```bash
basename /path/to/file
dirname /path/to/file
```

### Hard links

```bash
ln existing_file new_name
find . -inum INODE_NUMBER
```

### Symbolic links

```bash
ln -s TARGET LINK_NAME
readlink LINK_NAME
realpath LINK_NAME
```

### Find links

```bash
find . -type l
find . -xtype l
```

### Remove a link

```bash
rm link_name
unlink link_name
```

### Core mental models

```text
Hard link:
name A ---+
          +--> same inode --> same data
name B ---+

Symbolic link:
link inode --> stored pathname --> target pathname --> target inode
```

### Most important rules

1. The inode does not store the ordinary filename.
2. A directory maps names to inode numbers.
3. Hard links share one inode.
4. Symlinks store pathnames.
5. Hard links cannot cross filesystem boundaries.
6. Symlinks can cross filesystem boundaries.
7. Removing one hard link does not remove the other names.
8. Removing a symlink does not remove its target.
9. Removing a symlink target leaves a broken symlink.
10. `rm` removes a directory entry; storage reclamation depends on link count and open file descriptors.